# Phase 2 — Retrieval Index Building Notebook
**Hallucination Detection & Repair in LLM-Generated Answers**

This notebook downloads the FEVER Wikipedia corpus (Tier A), extracts passages, and builds:
1. **BM25 index** (saved as a pickle file)
2. **Dense FAISS index** using `BAAI/bge-small-en-v1.5` embeddings (saved to Google Drive in shards so disconnects won't lose work)

In [ ]:
# 1. Mount Google Drive and set directories
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE = Path("/content/drive/MyDrive/hallucination_project")
INDEX_DIR = DRIVE_BASE / "indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f"Indexes will be saved to: {INDEX_DIR}")

In [ ]:
# 2. Clone repo and install requirements
!git clone https://github.com/YOUR_USERNAME/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git repo
%cd repo
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm

In [ ]:
# 3. Download FEVER Wikipedia dump (Tier A)
# Downloading to local /tmp to save Drive bandwidth, then processed index is stored to Drive
!mkdir -p data/raw/fever
!wget -q --show-progress https://s3-eu-west-1.amazonaws.com/fever.ai/wiki-pages.zip -O /tmp/wiki-pages.zip
!unzip -q /tmp/wiki-pages.zip -d data/raw/fever/
!ls -lh data/raw/fever/wiki-pages | head -n 5

In [ ]:
# 4. Build BM25 and FAISS index (saving to Drive)
!python scripts/build_index.py \
    --tier A \
    --output_dir {INDEX_DIR} \
    --encoder BAAI/bge-small-en-v1.5 \
    --max_passages 1500000

In [ ]:
# 5. Smoke test retrieval
import pickle
from src.retrieve import retrieve_bm25

test_claim = "Albert Einstein was awarded the Nobel Prize in Physics in 1921."
passages = retrieve_bm25(test_claim, str(INDEX_DIR / "bm25_A.pkl"), top_k=3)

print(f"Query: {test_claim}\n")
for i, p in enumerate(passages):
    print(f"[{i+1}] (Score: {p.score:.3f}) {p.title}")
    print(f"    {p.text[:150]}...\n")